# trapX LLM Advisory — **`advisory_any_bar.py`** (any date + bar)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Chanakya1998begin/rdp/blob/main/advisory_any_bar_colab_v2.ipynb)

**Latest version.** This notebook runs the *actual* standalone `advisory_any_bar.py` tool — not a re-implementation. Given a **DATE** and **BAR_TIME** it:

1. Downloads that day's folder from the shared Google Drive (`gdown`, public, no Drive API) — the `llm_advisory_<date>.jsonl`, the trapX SQLite checkpoint, the day CSVs, and (optionally) the day's Postgres snapshot `pg_snapshot_<date>.db`.
2. **Gates** on the jsonl: stops if no pattern fired that minute.
3. Rebuilds the advisory input fresh (state-memory @ bar−1, the engine-computed snapshot per fired touchpoint, the minute's market data).
4. Fires **one converged LLM-advisory call** and prints the verdict.

The script, all skill prompts, **and a minimal `project/` package** (the two pure engine functions needed to recompute the v5 layer) are **embedded below** (base64) — the notebook is fully self-contained.

**Log parity with local.** When a `pg_snapshot_<date>.db` (produced on the trapX box with `_export_pg_day_snapshot.py`) is present in the day folder, the notebook feeds it to `--pg-snapshot` and the resulting log is **byte-identical** to a real-Postgres local replay — closing the CSV-only gap (run-cumulative floor/ceiling TRAP, option price-symmetry, jerk-family windowed OI). When the snapshot is absent, the notebook falls back to CSV-only (older behaviour) and prints a divergence disclaimer.

**Backends:** `nvidia` (default, needs `NVIDIA_API_KEY`) or `anthropic` / `auto` (live-parity, needs `ANTHROPIC_API_KEY`).

**Not in Colab:** the Breeze 1-second futures feed. If the requested bar fires `topbottom_formation`, that one touchpoint's 1-sec sustain reads as unavailable in the log — the verdict still runs, the caveat is explicit.

## 1. Install deps + load API keys
Keys are read from Colab **Secrets** (🔑 left sidebar → add `NVIDIA_API_KEY` and/or `ANTHROPIC_API_KEY`, toggle notebook access). Set at least the one matching your chosen backend.

In [ ]:
!pip install -q gdown openai anthropic python-dotenv langgraph langgraph-checkpoint-sqlite

import os
try:
    from google.colab import userdata
    for _k in ('NVIDIA_API_KEY', 'ANTHROPIC_API_KEY'):
        try:
            _v = userdata.get(_k)
        except Exception:
            _v = None
        if _v:
            os.environ[_k] = _v.strip()
except Exception:
    pass

print('NVIDIA_API_KEY   :', 'set' if os.environ.get('NVIDIA_API_KEY') else 'MISSING')
print('ANTHROPIC_API_KEY:', 'set' if os.environ.get('ANTHROPIC_API_KEY') else 'MISSING')

## 2. Write the embedded `advisory_any_bar.py` + `skills/` + `project/` to disk
Decodes the base64 payloads into `/content/`. The minimal `project/` package (`trapx_agent._audit_compute_v5_flags`, `llm_advisory.advisory._build_opening_audit_user_message`) lets the script **recompute the v5 layer** — essential because the recorded jsonl carries no `v5_*` fields (the live engine logs the raw facts; v5 is derived).

In [ ]:
import base64, json, pathlib

SCRIPT_B64 = "..."
SKILLS_B64 = "..."
PROJECT_B64 = "..."

pathlib.Path('advisory_any_bar.py').write_bytes(base64.b64decode(SCRIPT_B64))
skills = json.loads(base64.b64decode(SKILLS_B64).decode('utf-8'))
sd = pathlib.Path('skills'); sd.mkdir(exist_ok=True)
for name, text in skills.items():
    (sd / name).write_text(text, encoding='utf-8')
proj = json.loads(base64.b64decode(PROJECT_B64).decode('utf-8'))
for relpath, text in proj.items():
    p = pathlib.Path(relpath); p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(text, encoding='utf-8')
print('wrote advisory_any_bar.py +', len(skills), 'skills +', len(proj),
      'project files (engine v5 recompute enabled)')

## 3. Choose the bar + backend
`WHEN` is the bar in the script's native `DD-mon, HH:MM` form. `BACKEND`: `nvidia` (default), `anthropic`, or `auto` (pins to the backend/model the live engine recorded — like-for-like replay).

In [ ]:
#@title Parameters { run: "auto" }
WHEN    = "12-Jun, 09:19"  #@param {type:"string"}
BACKEND = "nvidia"         #@param ["nvidia", "anthropic", "auto"]
YEAR    = 2026             #@param {type:"integer"}
# Optional: model override (blank = backend default).
MODEL   = ""               #@param {type:"string"}
print('WHEN   =', WHEN)
print('BACKEND=', BACKEND, '  YEAR=', YEAR, '  MODEL=', MODEL or '(default)')

## 4. Mount Google Drive + resolve the day folder
Mounts your Drive and resolves the **day folder** for the chosen bar (`VM-4-logs/<Mon_DD>`, e.g. `Jun_12`). The notebook reuses the script's own date parser so the folder name always matches.

In [ ]:
#@title Drive folder { run: "auto" }
VM_LOGS_DIR = "/content/drive/MyDrive/VM-4-logs"  #@param {type:"string"}

from google.colab import drive
drive.mount('/content/drive')

import sys, os, glob, argparse
sys.path.insert(0, '.')
import advisory_any_bar as aab           # the script we wrote in step 2

# Reuse the script's exact parser so the Drive folder name always matches.
req = aab.parse_request(argparse.Namespace(when=WHEN, date=None,
                                           time=None, year=YEAR))
FOLDER_NAME = req.mon_dd                  # e.g. 'Jun_12'
DAY_DIR  = f'{VM_LOGS_DIR}/{FOLDER_NAME}'
LOG_NAME = f"advisory_{req.yyyymmdd}_{req.time.replace(':', '')}.log"
LOG_OUT  = f'{DAY_DIR}/{LOG_NAME}'
PG_SNAP  = f'{DAY_DIR}/pg_snapshot_{req.yyyymmdd}.db'

os.makedirs(DAY_DIR, exist_ok=True)
HAS_LOCAL = bool(glob.glob(f'{DAY_DIR}/llm_advisory_*.jsonl'))
HAS_PGSNAP = os.path.isfile(PG_SNAP)
print(f'date {req.iso_date}  ->  Drive folder {FOLDER_NAME!r}')
print('DAY_DIR :', DAY_DIR)
print('source  :', 'Drive day folder (--local-dir)' if HAS_LOCAL
      else 'gdown download (folder has no jsonl yet)')
print('pg_snap :',
      f'FOUND ({os.path.getsize(PG_SNAP)/1e6:.1f} MB) → full local-parity replay'
      if HAS_PGSNAP
      else 'MISSING → CSV-only replay (option-symmetry + run-cum floor/ceiling '
           'TRAP + jerk-family windowed OI will diverge from local; '
           'generate with `_export_pg_day_snapshot.py` on the trapX box and '
           f'drop `pg_snapshot_{req.yyyymmdd}.db` into this folder)')
print('log_out :', LOG_OUT)

## 5. Run the advisory — log saved back to the Drive folder
Invokes `python advisory_any_bar.py "<WHEN>" --skills-dir skills --backend <BACKEND> --no-live --out <DAY_DIR>/advisory_<date>_<time>.log`. The subprocess runs from `/content`, so it imports the embedded `project/` package and recomputes the v5 layer.

> **`--no-live` is required on Colab.** The script normally switches to a *live* path when the requested date is **today**, reading from a local Postgres that only exists on the trapX VM. `--no-live` forces the Drive/CSV path.

> **`--pg-snapshot` is added automatically** when `pg_snapshot_<date>.db` is present in the day folder (see step 4). That flag routes the six PG queries advisory issues to a sqlite copy of the day — matching the local log byte-for-byte. Without the snapshot the run still succeeds but is CSV-only.

In [ ]:
import subprocess, sys, shlex

cmd = [sys.executable, 'advisory_any_bar.py', WHEN,
       '--skills-dir', 'skills', '--backend', BACKEND, '--year', str(YEAR),
       '--no-live', '--out', LOG_OUT]
if HAS_LOCAL:
    cmd += ['--local-dir', DAY_DIR]
if HAS_PGSNAP:
    cmd += ['--pg-snapshot', PG_SNAP]
if MODEL.strip():
    cmd += ['--model', MODEL.strip()]
print('$', ' '.join(shlex.quote(c) for c in cmd), '\n')

proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stderr)
print('=' * 70)
print(proc.stdout)
if proc.returncode != 0:
    print('\n[exit code]', proc.returncode)

## 6. Confirm the log on Drive + show it
Lists the verbose log(s) now in the Drive day folder and prints the newest.

In [ ]:
import glob, os
logs = sorted(glob.glob(f'{DAY_DIR}/advisory_{req.yyyymmdd}_*.log'),
              key=os.path.getmtime)
if logs:
    print(f'{len(logs)} verbose log(s) saved in Drive folder {DAY_DIR}:')
    for p in logs:
        print('   ', os.path.basename(p), f'({os.path.getsize(p)} bytes)')
    print('\n===== newest:', os.path.basename(logs[-1]), '=====\n')
    print(open(logs[-1], encoding='utf-8').read())
else:
    print('no verbose log in', DAY_DIR,
          '— did the run gate out (no pattern at that bar) or error?')